<a href="https://colab.research.google.com/github/RaihahMahmud/FlyRank-AI--starter-ML-Internship-/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Rule

I rank content pages for review using observed Google Search Console signals. A page receives a higher priority when it has meaningful search visibility but an observed CTR opportunity. Pages without a strong signal receive a lower score and are marked for monitoring.

### Reason codes

* **CTR_OPPORTUNITY** — The page has observed search visibility and a low CTR relative to its search impressions.
* **NO_STRONG_SIGNAL** — The available signals do not provide a strong reason to prioritize the page.

### Action labels

* **REVIEW** — Prioritize the page for human review.
* **MONITOR** — No strong signal for immediate prioritization.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [26]:

import os
import pandas as pd

query = """
WITH page_level AS (
    SELECT
        content_hash_id,

        SUM(COALESCE(gsc_impressions, 0)) AS gsc_impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS gsc_clicks,

        CASE
            WHEN SUM(COALESCE(gsc_impressions, 0)) > 0
            THEN
                SUM(COALESCE(gsc_clicks, 0))::DOUBLE
                / SUM(COALESCE(gsc_impressions, 0))
            ELSE 0
        END AS ctr,

        AVG(gsc_avg_position) AS gsc_avg_position

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/month=2026-06/data_0.parquet'
    )

    GROUP BY content_hash_id
),

scored AS (
    SELECT
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        ctr,
        gsc_avg_position,

        CASE
            WHEN gsc_impressions >= 1000
                 AND gsc_avg_position <= 10
                 AND ctr < 0.05
            THEN 1
            ELSE 0
        END AS ctr_opportunity

    FROM page_level
)

SELECT
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    ctr,
    gsc_avg_position,
    ctr_opportunity,

    CASE
        WHEN ctr_opportunity = 1 THEN 1
        ELSE 0
    END AS priority_score,

    CASE
        WHEN ctr_opportunity = 1
            THEN 'CTR_OPPORTUNITY'
        ELSE 'NO_STRONG_SIGNAL'
    END AS reason_code,

    CASE
        WHEN ctr_opportunity = 1
            THEN 'REVIEW'
        ELSE 'MONITOR'
    END AS action

FROM scored

ORDER BY priority_score DESC, gsc_impressions DESC
"""

baseline_queue = con.sql(query).df()

baseline_queue["rank"] = range(1, len(baseline_queue) + 1)

print("Unique content pages:", len(baseline_queue))
display(baseline_queue.head(20))

Unique content pages: 409205


,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,ctr_opportunity,priority_score,reason_code,action,rank
0,content_963de14b1f58978f,615012.0,1676.0,0.002725,6.397556,1,1,CTR_OPPORTUNITY,REVIEW,1
1,content_eadb33b5df496f4a,591696.0,3817.0,0.006451,2.258612,1,1,CTR_OPPORTUNITY,REVIEW,2
2,content_545bb6cc7081ded3,585712.0,3048.0,0.005204,2.110776,1,1,CTR_OPPORTUNITY,REVIEW,3
3,content_943dc881428182b8,292416.0,399.0,0.001364,2.689206,1,1,CTR_OPPORTUNITY,REVIEW,4
4,content_f88878f155e4838d,277353.0,2543.0,0.009169,5.034392,1,1,CTR_OPPORTUNITY,REVIEW,5
5,content_9ef3d7516483e665,269943.0,787.0,0.002915,2.098715,1,1,CTR_OPPORTUNITY,REVIEW,6
6,content_c1f764a2f362d1c3,235427.0,45.0,0.000191,7.132457,1,1,CTR_OPPORTUNITY,REVIEW,7
7,content_b902320872acab45,234902.0,303.0,0.001290,5.510291,1,1,CTR_OPPORTUNITY,REVIEW,8
8,content_cc26620b2cbb837f,225417.0,386.0,0.001712,6.562809,1,1,CTR_OPPORTUNITY,REVIEW,9
9,content_acbcc847f8996314,212758.0,252.0,0.001184,4.856749,1,1,CTR_OPPORTUNITY,REVIEW,10


In [27]:
# Write required CSV

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(output_path, index=False)

print("CSV written successfully:", output_path)
print("Rows written:", len(baseline_queue))

CSV written successfully: work/outputs/baseline_action_score.csv
Rows written: 409205


### Top-20 review

The top-ranked pages are prioritized because they have substantial observed search impressions, a low observed CTR, and an average search position within the top 10.

The rule treats these pages as CTR opportunities for human review. This is directional decision-support, not evidence that changing a page will cause higher traffic or rankings.

In [28]:
# Top-20 review

top20 = baseline_queue.head(20).copy()

top20["confidence_note"] = (
    "Directional priority based on observed impressions, CTR, "
    "and average search position."
)

top20["what_would_make_it_wrong"] = (
    "The observed signals may not capture the actual reason "
    "for the low CTR, so human review is required."
)

display(
    top20[
        [
            "rank",
            "content_hash_id",
            "action",
            "reason_code",
            "priority_score",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,rank,content_hash_id,action,reason_code,priority_score,confidence_note,what_would_make_it_wrong
0,1,content_963de14b1f58978f,REVIEW,CTR_OPPORTUNITY,1,Directional priority based on observed impress...,The observed signals may not capture the actua...
1,2,content_eadb33b5df496f4a,REVIEW,CTR_OPPORTUNITY,1,Directional priority based on observed impress...,The observed signals may not capture the actua...
2,3,content_545bb6cc7081ded3,REVIEW,CTR_OPPORTUNITY,1,Directional priority based on observed impress...,The observed signals may not capture the actua...
3,4,content_943dc881428182b8,REVIEW,CTR_OPPORTUNITY,1,Directional priority based on observed impress...,The observed signals may not capture the actua...
4,5,content_f88878f155e4838d,REVIEW,CTR_OPPORTUNITY,1,Directional priority based on observed impress...,The observed signals may not capture the actua...
5,6,content_9ef3d7516483e665,REVIEW,CTR_OPPORTUNITY,1,Directional priority based on observed impress...,The observed signals may not capture the actua...
6,7,content_c1f764a2f362d1c3,REVIEW,CTR_OPPORTUNITY,1,Directional priority based on observed impress...,The observed signals may not capture the actua...
7,8,content_b902320872acab45,REVIEW,CTR_OPPORTUNITY,1,Directional priority based on observed impress...,The observed signals may not capture the actua...
8,9,content_cc26620b2cbb837f,REVIEW,CTR_OPPORTUNITY,1,Directional priority based on observed impress...,The observed signals may not capture the actua...
9,10,content_acbcc847f8996314,REVIEW,CTR_OPPORTUNITY,1,Directional priority based on observed impress...,The observed signals may not capture the actua...


### 4.Weak picks + leakage check

### Weak picks

The lowest-scoring pages have no strong observed signal for immediate prioritization. They are therefore assigned the `NO_STRONG_SIGNAL` reason code and `MONITOR` action.

These are weak picks for prioritization because the available signals do not provide enough evidence to justify sending them to the top of the review queue.

### Leakage check

The baseline uses only observed signals available in the current data window.

* Future-window fields used: **NONE**
* Product flags used: **NONE**
* Label-derived fields used: **NONE**

The score is intended for directional decision-support, not as a claim that refreshing a page will improve future performance.



In [29]:
# Weak picks

print("Lowest-scoring examples:")

display(
    baseline_queue.tail(10)[
        [
            "rank",
            "content_hash_id",
            "priority_score",
            "reason_code",
            "action"
        ]
    ]
)

# Leakage check
print("\nLeakage check:")
print("Future-window fields used: NONE")
print("Product flags used: NONE")
print("Label-derived fields used: NONE")

Lowest-scoring examples:


,rank,content_hash_id,priority_score,reason_code,action
409195,409196,content_1d2fba4b1a7b4946,0,NO_STRONG_SIGNAL,MONITOR
409196,409197,content_da3d768d3fcfaa60,0,NO_STRONG_SIGNAL,MONITOR
409197,409198,content_52e2f72bb8a84f4b,0,NO_STRONG_SIGNAL,MONITOR
409198,409199,content_973eddd2ebf893b7,0,NO_STRONG_SIGNAL,MONITOR
409199,409200,content_0a9c3aeede6c976d,0,NO_STRONG_SIGNAL,MONITOR
409200,409201,content_93fe5d29d001f96c,0,NO_STRONG_SIGNAL,MONITOR
409201,409202,content_8d8de87e7ee82237,0,NO_STRONG_SIGNAL,MONITOR
409202,409203,content_0081bc21dc426617,0,NO_STRONG_SIGNAL,MONITOR
409203,409204,content_1967265026ec39ac,0,NO_STRONG_SIGNAL,MONITOR
409204,409205,content_4f2794982b1ba0a4,0,NO_STRONG_SIGNAL,MONITOR



Leakage check:
Future-window fields used: NONE
Product flags used: NONE
Label-derived fields used: NONE


## Self-check

- [x] Every section is filled with markdown and supporting code.
- [x] The notebook runs from top to bottom without errors.
- [x] No client names, URLs, or private queries are included.
- [x] Claims use careful decision-support language.
- [x] The baseline uses observable March 2026 signals only.
- [x] No future-window outcomes or product flags are used.
- [x] The ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [x] Notebook committed to `work/notebooks/w04_baseline_score.ipynb`.